In [0]:
show catalogs;

In [0]:
show schemas in samples;

In [0]:
show tables in samples.nyctaxi;

In [0]:
select * from samples.nyctaxi.trips;

In [0]:
%python
df = spark.table("samples.nyctaxi.trips")
df.display()
df.printSchema()

In [0]:
%python
df.groupBy("pickup_zip").count().orderBy("count", ascending=False).show()

In [0]:
%python
df.write.mode("overwrite").option("overwriteSchema", "true").partitionBy("pickup_zip").saveAsTable("nyctaxi.bronze.trips")

In [0]:
describe table extended nyctaxi.bronze.trips;

In [0]:
%python
df = spark.table("nyctaxi.bronze.trips")
df.printSchema()

In [0]:
%python
df.filter(df.dropoff_zip == 10020).display()

In [0]:
%python
df.filter((df.pickup_zip == 10014) & (df.dropoff_zip == 10019)).display()

In [0]:
show volumes in nyctaxi.bronze;

In [0]:
%fs
ls

In [0]:
%python
df = spark.read.text("/Volumes/nyctaxi/bronze/taxi/retail_db/customers")

In [0]:
%python
customers = spark.read.csv("/Volumes/nyctaxi/bronze/taxi/retail_db/customers")


In [0]:
%python
display(customers)

In [0]:
%python
import json

# schema_dict = spark.read.json("/Volumes/nyctaxi/bronze/taxi/retail_db/schemas.json").first().asDict()
schema_dict = spark.read \
    .option("multiline", "true") \
    .json("/Volumes/nyctaxi/bronze/taxi/retail_db/schemas.json") \
    .first() \
    .asDict()

In [0]:
%python
display(schema_dict.keys())

In [0]:
%python
for key in schema_dict.keys():
    if key == "customers":
        customer_schema = schema_dict.get(key)
        print(customer_schema)

In [0]:
%python
from pyspark.sql.types import *

def get_type(t):
    return {
        "integer": IntegerType(),
        "string": StringType(),
        "float": FloatType(),
        "timestamp": TimestampType()
    }.get(t.lower() if t else "", StringType())

# sort columns
cols = sorted(customer_schema, key=lambda x: x["column_position"])
# print(cols)

# build schema
schema = StructType([
    StructField(col["column_name"], get_type(col["data_type"]), True)
    for col in cols
])

# print(schema)

# for col in cols:
#     s_type = StructType([
#         StructField(col["column_name"], get_type(col["data_type"]), True)
#         ])
#     print(s_type)

types = [
        StructField(col["column_name"], get_type(col["data_type"]), True)
        for col in cols
        ]
schema = StructType(types)
print(schema)



In [0]:
%python
df = spark.read \
    .option("header", "false") \
    .schema(schema) \
    .csv("/Volumes/nyctaxi/bronze/taxi/retail_db/customers/")

df.display()

In [0]:
%python
{
        "integer": IntegerType(),
        "string": StringType(),
        "float": FloatType(),
        "timestamp": TimestampType()
    }.get("string")

In [0]:
%python
numbers = [1, 2, 3]
squares = [x*x for x in numbers]
print(list(squares))